In [ ]:
# UPAR27 Track1 PAR LARGE v7 | aaravsubedi | ViT-L/14 + best-mA + strong augHF_TOKEN = os.environ.get("HF_TOKEN", "hf_YOUR_TOKEN_HERE")  # was hardcoded during Kaggle runs; set HF_TOKEN env/secretHF_REPO="ShivRamSaud/upar-track1-par-large"import os, gc, time, jsonos.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'

In [ ]:
!pip install -q transformers huggingface_hub timm scikit-learn tqdm gdown
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(i, torch.cuda.get_device_name(i))

In [ ]:
from huggingface_hub import login, create_repo, snapshot_download, HfApi
login(token=HF_TOKEN)
create_repo(HF_REPO, exist_ok=True, private=False)
print('HF ok', HF_REPO)

In [ ]:
# Annotations (light, 1.7MB) + images via Kaggle datasets if attached, else download
!rm -rf /tmp/upar && git clone --depth 1 https://github.com/speckean/UPAR-Challenge-2027 /tmp/upar
!ls /kaggle/input 2>&1 | head -30
!ls /tmp/upar/data/annotations/task1/train 2>&1 | head
import pathlib
# Resolve image roots: try kaggle inputs first
CANDS=["/kaggle/input/market1501","/kaggle/input/pa100k","/kaggle/input/upar2027-dev/data","/tmp/upar/data"]
for c in CANDS+ ["/kaggle/input"]:
  p=pathlib.Path(c)
  if p.exists(): print(c, "->", [x.name for x in list(p.iterdir())[:10]])

In [ ]:
import torch.nn as nn
from transformers import CLIPModel, CLIPProcessor
MID='openai/clip-vit-large-patch14'
clip=CLIPModel.from_pretrained(MID)
try:
 clip.vision_model.gradient_checkpointing_enable()
except: pass
print('CLIP ok, image_size', clip.config.vision_config.image_size)
class CLIP_PAR(nn.Module):
 def __init__(self,clip,n=40):
  super().__init__(); self.clip=clip; d=clip.config.projection_dim
  self.head=nn.Sequential(nn.Dropout(0.3),nn.Linear(d,512),nn.GELU(),nn.Dropout(0.2),nn.Linear(512,n))
 def forward(self,pix):
  v=self.clip.get_image_features(pixel_values=pix)
  import torch as _t
  _o=v
  v=_o[0] if isinstance(_o,(tuple,list)) else (_o.pooler_output if hasattr(_o,'pooler_output') and _o.pooler_output is not None else (_o.last_hidden_state[:,0] if hasattr(_o,'last_hidden_state') else (_o.image_embeds if hasattr(_o,'image_embeds') else _o)))
  v=v if _t.is_tensor(v) else _t.as_tensor(v)
  return self.head(v)
model=CLIP_PAR(clip).cuda().train()
print(f'{sum(p.numel() for p in model.parameters())/1e6:.1f}M')

In [ ]:
Dataset with fast basename index (Market1501 + PA-100K, 89001/97670 train kept) + RandomErasing aug. Same structure as notebook_base_ran.ipynb with ColorJitter 0.25 + RandomErasing(p=0.25).

In [ ]:
Train: batch 16 x accum 4, AdamW (2e-5 backbone / 8e-5 head), ASL, AMP, best-val tracking to best_model.bin (restored as pytorch_model.bin at end). 5 epochs before 6.2h guard: ep0 0.9578, ep1 0.9567, ep2 0.9556, ep3 0.9555, ep4 0.9557. Final BEST pushed to ShivRamSaud/upar-track1-par-large.

In [ ]:
model.eval()
import csv
with torch.no_grad(), open('/tmp/pred_val.csv','w',newline='') as f:
 w=csv.writer(f)
 for i in range(min(len(va),5000)):
  x,_=va[i]; p=(torch.sigmoid(model(x.unsqueeze(0).cuda())).cpu().numpy()[0]>0.5).astype(int)
  w.writerow([va.rows[i][0]]+p.tolist())
HfApi().upload_file(repo_id=HF_REPO,path_in_repo='pred_val.csv',path_or_fileobj='/tmp/pred_val.csv',commit_message='preds')
print('preds on HF')